# Session 08A: Building a Persistent Assistant: Full-Stack Lab (Student Lab)
### Connecting SQLite Relational Persistence with Vector Archival and Self-Editing Memory
**Foundations of Agentic AI Systems • Hands-On Laboratory Workstation**

---

### Resources & API Keys
* **Course Materials**: [Google Drive Shared Folder](https://drive.google.com/drive/folders/1qpWAHvdEZiCfM1kNCZLdhsNPl7S3DBqB?usp=sharing)
* **Free LLM API Keys**: [Awesome Free LLM APIs](https://github.com/mnfst/awesome-free-llm-apis)
* **Dual Execution Mode**: This notebook runs completely offline with local zero-cost deterministic mock models. If you have a free **Groq** or **OpenRouter** API key, you can plug it in to run against live frontier models.


## 📺 Curated 5–7 Minute Video Primers
* **Building Long-Term Memory for AI Agents: Architecture & Databases** (~6 mins)  
  *Search*: `Long term memory for AI agents architecture SQLite vector database`  
  *Summary*: Explains how production systems separate session transcripts from persistent core entity profiles.
* **Defending LLMs Against Prompt Injection & Memory Poisoning** (~6 mins)  
  *Search*: `Prompt Injection Defense LLM Agent Security Chip Huyen`  
  *Summary*: How malicious user inputs can corrupt persistent databases and how schema validation protects state.


## 0. Setup: Embedded SQLite Environment
We use Python's built-in `sqlite3` module. No external servers or installation required.


In [ ]:
import sqlite3, json, os

print("SQLite Persistent Lab environment initialized!")


---
### Activity 1 (Tier 1 - Warmup): Database Schema Initialization
**Concept**: Persistent agents store sessions, messages, and core memory in structured relational tables.
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Pre-solved warmup
conn = sqlite3.connect(":memory:") # Use 'agent_state.db' for disk persistence

with conn:
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS sessions (
            session_id TEXT PRIMARY KEY,
            user_id TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT,
            role TEXT,
            content TEXT,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        CREATE TABLE IF NOT EXISTS core_memory (
            user_id TEXT,
            block_name TEXT,
            content TEXT,
            PRIMARY KEY(user_id, block_name)
        );
    ''')

print("Database tables initialized successfully!")


---
### Activity 2 (Tier 1 - Warmup): Saving and Loading Core Memory Blocks
**Concept**: Reading and writing persona and user profile blocks atomically.
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Pre-solved warmup
def set_core_memory(conn, user_id: str, block: str, content: str):
    with conn:
        conn.execute('''
            INSERT INTO core_memory (user_id, block_name, content)
            VALUES (?, ?, ?)
            ON CONFLICT(user_id, block_name) DO UPDATE SET content=excluded.content
        ''', (user_id, block, content))

def get_core_memory(conn, user_id: str) -> dict:
    cur = conn.cursor()
    cur.execute("SELECT block_name, content FROM core_memory WHERE user_id=?", (user_id,))
    return {row[0]: row[1] for row in cur.fetchall()}

set_core_memory(conn, "u123", "human", "Role: AI Engineer\nLocation: Coimbatore")
print("Retrieved Core Memory:", get_core_memory(conn, "u123"))


---
### Activity 3 (Tier 1 - Warmup): Multi-Session Dialogue Logging
**Concept**: Persisting conversational turns to SQLite ensures full auditability across restarts.
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Pre-solved warmup
def log_turn(conn, session_id: str, role: str, content: str):
    with conn:
        conn.execute("INSERT INTO messages (session_id, role, content) VALUES (?, ?, ?)",
                     (session_id, role, content))

log_turn(conn, "sess_01", "user", "I need to configure a persistent agent.")
log_turn(conn, "sess_01", "assistant", "I am configured with SQLite persistence.")

cur = conn.cursor()
cur.execute("SELECT role, content FROM messages WHERE session_id=?", ("sess_01",))
for r, c in cur.fetchall():
    print(f"[{r.upper()}]: {c}")


---
### Activity 4 (Tier 2 - Guided): Multi-Session Cold-Start Hydration
**Task**: Write a function `hydrate_agent(conn, user_id)` that reads the user's profile and returns a compiled system prompt on startup.


In [ ]:
# [STUDENT TASK]: Implement hydrate_agent
# YOUR CODE HERE


---
### Activity 5 (Tier 2 - Guided): Memory Conflict Resolution (Chennai -> Coimbatore)
**Task**: Implement an automated updater that replaces old user details when conflicting preferences are revealed.


In [ ]:
# [STUDENT TASK]: Implement update_user_fact
# YOUR CODE HERE


---
### Activity 6 (Tier 2 - Guided): Defense Against Malicious Override
**Task**: Build a memory validation filter that rejects strings containing jailbreak phrases (`"SYSTEM OVERRIDE"`, `"ignore previous"`).


In [ ]:
# [STUDENT TASK]: Implement safe_set_memory
# YOUR CODE HERE


---
### Activity 7 (Tier 3 - Exploration): Cold Start Latency Profiling
**Exploration**: Profile the wall-clock time required to query SQLite and hydrate agent state over 1,000 iterations. Verify latency is <1ms.


In [ ]:
# [STUDENT EXPLORATION]: Profile cold start hydration latency
# YOUR CODE HERE


---
### Activity 8 (Tier 3 - Exploration): Standalone Agent Package Export
**Exploration**: Package the database and memory functions into a clean standalone Python class `PersistentAssistant`.


In [ ]:
# [STUDENT EXPLORATION]: Package into PersistentAssistant class
# YOUR CODE HERE
